# EDA: labelled training rows (Person D)

Goal for 10:30–11:15: compare suspicious and legitimate transactions by **leg count**, **USD value** and **cross-currency**, then check the GPU and PyG for C.

Each section ends with a short note to pass on to A, B or C.

## 0. Setup (Colab)
Upload `decode-and-detect-training-data.zip` to the Colab file pane, or put it in Drive and change `ZIP` below.

In [ ]:
# from google.colab import drive; drive.mount('/content/drive')   # if the zip is in Drive
import os, zipfile
ZIP = '/content/decode-and-detect-training-data.zip'   # or '/content/drive/MyDrive/...'
ROOT = '/content/data'
if not os.path.exists(ROOT):
    zipfile.ZipFile(ZIP).extractall(ROOT)
for dirpath, _, files in os.walk(ROOT):
    if 'labels.csv' in files:
        TRAIN = dirpath
DATA = os.path.dirname(TRAIN)          # folder holding train/, validation/, payments.py
print(TRAIN)

### GPU and PyG check (for C)

In [ ]:
import torch
print('torch', torch.__version__, '| CUDA:', torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
try:
    import torch_geometric
except ImportError:
    !pip -q install torch_geometric
    import torch_geometric
print('PyG', torch_geometric.__version__)

## 1. Load and join

In [ ]:
import sys, json
import numpy as np, pandas as pd, matplotlib.pyplot as plt
sys.path.insert(0, DATA)
from payments import load_rates, price, XRP

tx  = pd.read_parquet(f'{TRAIN}/transactions.parquet')
tr  = pd.read_parquet(f'{TRAIN}/transfers.parquet')
lab = pd.read_csv(f'{TRAIN}/labels.csv')
rates = load_rates(TRAIN)
for name, df in [('transactions', tx), ('transfers', tr), ('labels', lab)]:
    print(f'{name:13s} {df.shape}')
assert tx.TRANSACTION_HASH.is_unique and set(tx.TRANSACTION_HASH) == set(lab.TRANSACTION_HASH)

**Value every leg in USD.** An asset is `(TOKEN_ADDRESS, SYMBOL)`: USD from one issuer is a different asset from USD from the other. `market_rates.csv` uses `XRP` or `issuer:currency`, priced for the hour containing the timestamp.

In [ ]:
tr['TS'] = tr.TIMESTAMP.astype(int)
tr['ASSET'] = np.where(tr.TOKEN_ADDRESS == XRP, 'XRP', tr.TOKEN_ADDRESS + ':' + tr.SYMBOL)
tr['USD'] = [v * price(rates, t, a) / 1e6 for v, t, a in zip(tr.VALUE, tr.TS, tr.ASSET)]

tx['TS'] = tx.TIMESTAMP.astype(int)
extra = tx.EXTRA_TRANSACTION_FIELDS.map(json.loads)
tx['DEST'] = extra.map(lambda f: f['Destination'])
tx['HAS_SENDMAX'] = extra.map(lambda f: 'SendMax' in f)
tx['HAS_PATHS'] = extra.map(lambda f: 'Paths' in f)

legs = tr.groupby('TRANSACTION_HASH').agg(
    n_legs=('TRANSFER_INDEX', 'size'),
    n_assets=('ASSET', 'nunique'),
    n_symbols=('SYMBOL', 'nunique'),
    usd_max=('USD', 'max'),
    usd_first=('USD', 'first'),      # first executed leg = actual source cost
    usd_last=('USD', 'last'),        # last leg = what the destination received
)
legs['cross_currency'] = legs.n_assets > 1
legs['slippage'] = 1 - legs.usd_last / legs.usd_first

df = (tx[['TRANSACTION_HASH', 'TS', 'DATETIME', 'ACCOUNT', 'DEST', 'HAS_SENDMAX', 'HAS_PATHS']]
      .merge(legs, left_on='TRANSACTION_HASH', right_index=True)
      .merge(lab, on='TRANSACTION_HASH'))
df.head()

## 2. Labels: what we are predicting

In [ ]:
print(df.is_suspicious.value_counts(), f"\npositive rate {df.is_suspicious.mean():.2%}")
print(pd.crosstab(df.family, [df.difficulty]).assign(total=lambda x: x.sum(1)).sort_values('total', ascending=False))

Things to note here:

- About **3.1% positive** (1,250 of 40,131). A random ranking gets AP ≈ 0.031, so treat that as the floor.
- `family` is **not** the same as the label. Most families have both suspicious rows and legitimate look-alikes (`aggregate_relay` has 102 positives among ~10k rows). The model has to separate a pattern from its decoy.
- `family`, `difficulty`, `scenario_id` and `cohort_id` are **annotation only** and don't exist in test. Never use them as features. They're fine for grouping CV folds and for error analysis.
- The 25 `ambiguous` positives are random by design, so no model will catch them.

## 3. Suspicious vs legit by leg count

In [ ]:
def rate_table(col):
    t = df.groupby(col).is_suspicious.agg(n='size', suspicious='sum', rate='mean')
    t['lift'] = t.rate / df.is_suspicious.mean()
    return t.round(3)
rate_table('n_legs')

In [ ]:
pd.crosstab(df.n_legs, df.difficulty, normalize='columns').round(3)

## 4. Suspicious vs legit by USD value

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
bins = np.logspace(-2, 4, 60)
for y, name in [(0, 'legit'), (1, 'suspicious')]:
    axes[0].hist(df.loc[df.is_suspicious == y, 'usd_max'], bins=bins, density=True, alpha=.6, label=name)
axes[0].set_xscale('log'); axes[0].set_xlabel('max leg USD'); axes[0].legend(); axes[0].set_title('USD value by label')
df['usd_bin'] = pd.qcut(df.usd_max, 10)
rate_table('usd_bin').rate.plot.bar(ax=axes[1], title='suspicious rate by USD decile')
axes[1].axhline(df.is_suspicious.mean(), ls='--', c='k'); plt.tight_layout()
df.groupby('difficulty').usd_max.describe(percentiles=[.1, .5, .9]).round(1)

In [ ]:
# round amounts in the delivered asset's units (last leg)
last = df.TRANSACTION_HASH.map(tr.sort_values('TRANSFER_INDEX').groupby('TRANSACTION_HASH').VALUE.last()).round(6)
for step in [100, 10, 1, 0.01]:
    print(f'multiple of {step:>6}: {(last % step).round(6).eq(0).sum()} txs')
# expect ~0 everywhere: amounts carry 6 noisy decimals -> tell B to drop 'round amounts'

## 5. Cross-currency

In [ ]:
display(rate_table('cross_currency'))
display(rate_table('n_assets'))
display(df.groupby('is_suspicious').slippage.describe().round(4))

In [ ]:
# which asset pairs (first leg -> last leg)
ends = tr.sort_values('TRANSFER_INDEX').groupby('TRANSACTION_HASH').ASSET.agg(['first', 'last'])
short = lambda a: a if a == 'XRP' else a.split(':')[0][:5] + ':' + a.split(':')[1]
df['pair'] = df.TRANSACTION_HASH.map(ends['first'].map(short) + ' -> ' + ends['last'].map(short))
rate_table('pair').sort_values('n', ascending=False)

### Checkpoint: what sections 3–5 say

On this data, **leg count, cross-currency, asset pair and round amounts barely move the suspicious rate** (everything sits around 3%, lift ≈ 1). The README said as much: *"look at aggregate flows and context, not just isolated payment amounts or fixed routes."* USD value shifts a little (suspicious rows run a bit larger), so it's a weak feature.

**Tell the team:** per-transaction leg features alone will give a weak GBDT. The signal is in **accounts, time and graph structure**, which backs up C's GNN plan and B's account aggregates. The next sections show where that signal is.

## 6. Where the signal is: accounts and scenarios

In [ ]:
s = df[df.is_suspicious == 1]
print('accounts total      ', pd.unique(df[['ACCOUNT', 'DEST']].values.ravel()).size)
print('suspicious senders  ', s.ACCOUNT.nunique(), '| receivers', s.DEST.nunique())
print('txs per scenario    ', s.groupby('scenario_id').size().describe().round(1).to_dict())

# per-account share of sent txs that are suspicious
acc = df.groupby('ACCOUNT').is_suspicious.agg(['size', 'mean'])
print((acc['mean'] > 0).mean(), 'of senders have >=1 suspicious tx')
acc['mean'].plot.hist(bins=50, logy=True, title='per-sender suspicious share');

If most senders were either all-legit or mostly-suspicious, account identity alone would carry the label. Here it doesn't: only ~700 accounts, each with ~40–300 txs, and about two-thirds of senders have at least one suspicious tx. Suspicious txs come from busy accounts that are mostly legitimate, so **labels live on transactions, not accounts**. That supports C's tx-node bipartite design over account-node classification. Degree is also similar for both classes (suspicious slightly lower).

In [ ]:
# degree of the accounts involved: are suspicious txs between low-activity or hub accounts?
deg = pd.concat([df.ACCOUNT, df.DEST]).value_counts()
df['src_deg'] = df.ACCOUNT.map(deg); df['dst_deg'] = df.DEST.map(deg)
df.groupby('is_suspicious')[['src_deg', 'dst_deg']].describe(percentiles=[.1, .5, .9]).T.round(0)

## 7. Time: pass-through and bursts

In [ ]:
# pass-through: receiver sends money on within 1h of receiving it
df = df.sort_values('TS').reset_index(drop=True)
out_ts = df.groupby('ACCOUNT').TS.apply(np.sort)
def forwards_within(dest, ts, window=3600):
    t = out_ts.get(dest)
    if t is None: return False
    i = np.searchsorted(t, ts, side='right')
    return i < len(t) and t[i] - ts <= window
df['dest_forwards_1h'] = [forwards_within(d, t) for d, t in zip(df.DEST, df.TS)]
rate_table('dest_forwards_1h')

In [ ]:
# sender burstiness: txs from the same sender in the surrounding 10 minutes
df['sender_10m'] = df.groupby('ACCOUNT').TS.transform(
    lambda t: np.searchsorted(t.values, t.values + 300, 'right') - np.searchsorted(t.values, t.values - 300, 'left'))
df['sender_10m_bin'] = pd.cut(df.sender_10m, [0, 1, 2, 5, 10, 1e9])
rate_table('sender_10m_bin')

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
daily = df.set_index('DATETIME').resample('6h').is_suspicious.agg(['size', 'mean'])
daily['mean'].plot(ax=ax[0], title='suspicious rate per 6h'); ax[0].axhline(df.is_suspicious.mean(), ls='--', c='k')
df.groupby(df.DATETIME.dt.hour).is_suspicious.mean().plot.bar(ax=ax[1], title='suspicious rate by hour of day'); plt.tight_layout()

Expect **sender bursts** to be the strongest single signal: a sender with 6–10 txs in a 10-minute window is ~90% suspicious, and 3–5 txs is ~15% (5× lift). Pass-through within 1h gives ~1.5× lift. These are temporal-neighbourhood features, so the GNN needs time information on its edges or nodes.

**For C and A (split rule):** the positive rate is **not flat over the week**. Day 2 runs about twice the average. A time-based split has to put enough positives in validation, so check the counts:

In [ ]:
cut = df.TS.quantile(0.8)
for name, part in [('train (first 80% of time)', df[df.TS <= cut]), ('val (last 20%)', df[df.TS > cut])]:
    print(f'{name:28s} n={len(part):6d}  pos={part.is_suspicious.sum():4d}  rate={part.is_suspicious.mean():.3%}')
# scenarios that straddle the cut leak context across the split
sc = df[df.is_suspicious == 1].groupby('scenario_id').TS.agg(['min', 'max'])
print('scenarios crossing the cut:', ((sc['min'] <= cut) & (sc['max'] > cut)).sum(), 'of', len(sc))

## 8. Quick feature-ranking check

In [ ]:
from sklearn.metrics import average_precision_score as ap
feats = ['n_legs', 'n_assets', 'usd_max', 'slippage', 'HAS_PATHS',
         'src_deg', 'dst_deg', 'dest_forwards_1h', 'sender_10m']
out = {f: ap(df.is_suspicious, df[f].astype(float).fillna(0)) for f in feats}
out |= {f + ' (neg)': ap(df.is_suspicious, -df[f].astype(float).fillna(0)) for f in feats}
pd.Series(out).sort_values(ascending=False).to_frame('AP (baseline ≈ 0.031)').round(4)

Single-feature AP tells you which raw signals beat random (0.031) before any model sees them. Send the top ones to B and A.

## 9. Summary to post in the team chat

Fill this in from the outputs above:

- Positive rate ___ %, AP floor ≈ 0.031. `family`/`difficulty`/`scenario_id` are annotation only, not features.
- Leg count / cross-currency / asset pair / round amounts: lift ≈ ___ → weak on their own.
- USD: suspicious median ___ vs legit ___ → weak.
- Strongest raw signals: ___ (pass-through, burst, degree).
- Time split: val positives ___, scenarios crossing the cut ___. Consider grouping by `scenario_id` when building folds.
- GPU: ___, PyG: ___.